# Analyse de la transition énergétique (2000-2026)

## Questions métier
1. **Mix énergétique** : quels pays ont le plus réduit la part du charbon et le plus augmenté celle du solaire et de l'éolien depuis 2000 ?
2. **Mix et émissions** : les pays qui ont le plus augmenté leur part de renouvelables ont-ils réduit leurs émissions de CO2 par habitant ?
3. **Prix du carbone** : comment évolue le prix par marché, mois par mois, et où sont les ruptures (par exemple la crise énergétique de 2022) ?
4. **Température et CO2** : l'anomalie de température globale suit-elle le CO2 atmosphérique, et quelles régions se réchauffent le plus vite ?
5. **Événements climatiques** : les événements les plus graves surviennent-ils dans les années les plus chaudes, et la gravité dépend-elle du type d'événement ?

Source : Kaggle, *Global Climate & Energy Transition 2000-2026* (licence CC0).

## 1. Chargement et typage
Les colonnes de dates sont lues en texte par défaut : on les type dès la lecture avec `parse_dates`.

In [3]:
from pathlib import Path
import pandas as pd

DOSSIER_RAW = Path("../data/raw")

prix_carbone = pd.read_csv(DOSSIER_RAW / "carbon_prices_daily.csv", parse_dates=["date"])
evenements = pd.read_csv(DOSSIER_RAW / "climate_events.csv", parse_dates=["date"])
emissions = pd.read_csv(DOSSIER_RAW / "co2_emissions_yearly.csv")
mix_energie = pd.read_csv(DOSSIER_RAW / "energy_mix_yearly.csv")
temperature = pd.read_csv(DOSSIER_RAW / "temperature_anomaly_monthly.csv", parse_dates=["year_month"])

donnees = {
    "prix_carbone": prix_carbone,
    "evenements": evenements,
    "emissions": emissions,
    "mix_energie": mix_energie,
    "temperature": temperature,
}

## 2. Inspection générale

In [4]:
resume = pd.DataFrame({
    nom: {
        "lignes": df.shape[0],
        "colonnes": df.shape[1],
        "manquants": int(df.isna().sum().sum()),
        "doublons": int(df.duplicated().sum()),
    }
    for nom, df in donnees.items()
}).T
resume

,lignes,colonnes,manquants,doublons
prix_carbone,15866,5,0,0
evenements,50,11,0,0
emissions,1350,8,0,0
mix_energie,1350,14,0,0
temperature,2528,7,2212,0


In [5]:
for nom, df in donnees.items():
    print(f"===== {nom} =====")
    print("Types :", dict(df.dtypes.astype(str)))
    manquants = df.isna().sum()
    print("Manquants :", manquants[manquants > 0].to_dict() or "aucun")
    print()

===== prix_carbone =====
Types : {'date': 'datetime64[us]', 'year': 'int64', 'market': 'str', 'currency': 'str', 'price': 'float64'}
Manquants : aucun

===== evenements =====
Types : {'event_id': 'str', 'date': 'datetime64[us]', 'year': 'int64', 'month': 'int64', 'region': 'str', 'event_type': 'str', 'severity_score': 'int64', 'description': 'str', 'is_policy': 'int64', 'is_extreme_weather': 'int64', 'is_disaster': 'int64'}
Manquants : aucun

===== emissions =====
Types : {'year': 'int64', 'country': 'str', 'iso3': 'str', 'region': 'str', 'co2_emissions_mt': 'float64', 'population_millions': 'float64', 'co2_per_capita_t': 'float64', 'co2_intensity_kg_per_gdp_usd': 'float64'}
Manquants : aucun

===== mix_energie =====
Types : {'year': 'int64', 'country': 'str', 'iso3': 'str', 'region': 'str', 'coal_pct': 'float64', 'oil_pct': 'float64', 'gas_pct': 'float64', 'nuclear_pct': 'float64', 'hydro_pct': 'float64', 'solar_pct': 'float64', 'wind_pct': 'float64', 'other_renewables_pct': 'float64'

In [16]:
for nom, df in donnees.items():
    print(f"===== {nom} =====")
    df.info()
    print()

===== prix_carbone =====
<class 'pandas.DataFrame'>
RangeIndex: 15866 entries, 0 to 15865
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      15866 non-null  datetime64[us]
 1   year      15866 non-null  int64         
 2   market    15866 non-null  str           
 3   currency  15866 non-null  str           
 4   price     15866 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 619.9 KB

===== evenements =====
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   event_id            50 non-null     str           
 1   date                50 non-null     datetime64[us]
 2   year                50 non-null     int64         
 3   month               50 non-null     int64         
 4   region              

In [17]:
for nom, df in donnees.items():
    print(f"===== {nom} =====")
    display(df.describe())

===== prix_carbone =====


,date,year,price
count,15866,15866.000000,15866.00000
mean,2018-06-26 19:30:48.329761,2017.986701,27.14063
min,2005-04-22 00:00:00,2005.000000,0.05000
25%,2014-05-16 18:00:00,2014.000000,6.36000
50%,2019-06-11 12:00:00,2019.000000,15.50500
75%,2023-04-17 00:00:00,2023.000000,35.19000
max,2026-04-30 00:00:00,2026.000000,119.52000
std,NaN,5.588788,28.23845


===== evenements =====


,date,year,month,severity_score,is_policy,is_extreme_weather,is_disaster
count,50,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,2019-05-25 16:19:12,2018.820000,7.460000,7.920000,0.220000,0.620000,0.100000
min,2003-08-01 00:00:00,2003.000000,1.000000,6.000000,0.000000,0.000000,0.000000
25%,2016-08-17 06:00:00,2016.250000,5.250000,7.000000,0.000000,0.000000,0.000000
50%,2021-04-21 12:00:00,2021.000000,8.000000,8.000000,0.000000,1.000000,0.000000
75%,2023-11-19 12:00:00,2023.000000,10.000000,8.000000,0.000000,1.000000,0.000000
max,2026-04-10 00:00:00,2026.000000,12.000000,10.000000,1.000000,1.000000,1.000000
std,NaN,6.076687,3.252691,0.778276,0.418452,0.490314,0.303046


===== emissions =====


,year,co2_emissions_mt,population_millions,co2_per_capita_t,co2_intensity_kg_per_gdp_usd
count,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000
mean,2013.000000,602.218444,119.570296,7.685822,0.767867
std,7.791767,1466.561207,257.430643,6.472427,0.726741
min,2000.000000,11.000000,2.500000,0.180000,0.030000
25%,2006.000000,99.325000,19.400000,2.980000,0.240000
50%,2013.000000,227.050000,46.300000,6.420000,0.520000
75%,2020.000000,444.800000,102.550000,10.675000,1.087500
max,2026.000000,12218.200000,1464.600000,40.300000,4.300000


===== mix_energie =====


,year,coal_pct,oil_pct,gas_pct,nuclear_pct,hydro_pct,solar_pct,wind_pct,other_renewables_pct,renewables_total_pct,fossil_total_pct
count,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000
mean,2013.000000,15.358970,36.529304,25.293719,5.157615,7.880600,2.973963,2.747970,4.057570,17.660215,77.182059
std,7.791767,15.737703,7.962231,13.641831,6.444548,8.437632,2.552514,2.887144,3.022374,11.956569,12.704714
min,2000.000000,0.000000,10.780000,2.640000,0.000000,0.000000,0.000000,0.000000,0.000000,0.670000,40.810000
25%,2006.000000,3.817500,32.500000,14.390000,1.150000,2.100000,0.875000,0.460000,1.690000,7.432500,67.642500
50%,2013.000000,11.865000,35.500000,24.120000,2.545000,4.740000,2.520000,1.825000,3.860000,15.655000,77.820000
75%,2020.000000,23.890000,42.912500,37.357500,8.977500,9.935000,4.567500,4.130000,5.610000,23.397500,89.565000
max,2026.000000,75.450000,51.840000,56.120000,42.470000,35.280000,18.840000,15.370000,15.190000,51.490000,99.330000


===== temperature =====


,year_month,year,month,temp_anomaly_c,co2_ppm
count,2528,2528.000000,2528.000000,2528.000000,316.00000
mean,2013-02-14 14:25:49.367088,2012.670886,6.449367,1.125491,398.31519
min,2000-01-01 00:00:00,2000.000000,1.000000,0.219000,365.78000
25%,2006-07-24 06:00:00,2006.000000,3.000000,0.641750,381.07250
50%,2013-02-15 00:00:00,2013.000000,6.000000,0.902000,394.32000
75%,2019-09-08 12:00:00,2019.000000,9.000000,1.316250,415.88250
max,2026-04-01 00:00:00,2026.000000,12.000000,4.522000,436.88000
std,NaN,7.605170,3.462132,0.753221,19.77562


In [6]:
for nom, df in donnees.items():
    print(f"===== {nom} =====")
    display(df.head(3))

===== prix_carbone =====


,date,year,market,currency,price
0,2005-04-22,2005,EU_ETS,EUR,7.17
1,2005-04-25,2005,EU_ETS,EUR,7.13
2,2005-04-26,2005,EU_ETS,EUR,7.44


===== evenements =====


,event_id,date,year,month,region,event_type,severity_score,description,is_policy,is_extreme_weather,is_disaster
0,CE001,2003-08-01,2003,8,Europe,heatwave,8,"European heatwave kills 70,000+",0,1,0
1,CE002,2004-12-26,2004,12,Asia,tsunami,9,"Indian Ocean tsunami (227,000 deaths)",0,0,1
2,CE003,2005-08-29,2005,8,USA,hurricane,9,"Hurricane Katrina (~1,800 deaths, $125B)",0,1,0


===== emissions =====


,year,country,iso3,region,co2_emissions_mt,population_millions,co2_per_capita_t,co2_intensity_kg_per_gdp_usd
0,2000,China,CHN,Asia,3535.5,1198.5,2.95,0.59
1,2001,China,CHN,Asia,4049.2,1207.9,3.35,0.61
2,2002,China,CHN,Asia,4601.0,1217.4,3.78,0.63


===== mix_energie =====


,year,country,iso3,region,coal_pct,oil_pct,gas_pct,nuclear_pct,hydro_pct,solar_pct,wind_pct,other_renewables_pct,renewables_total_pct,fossil_total_pct
0,2000,China,CHN,Asia,68.00,21.98,2.78,0.87,6.36,0.01,0.00,0.00,6.37,92.75
1,2001,China,CHN,Asia,67.64,21.35,3.36,0.97,6.46,0.00,0.04,0.17,6.68,92.36
2,2002,China,CHN,Asia,67.56,21.21,3.19,1.30,6.38,0.00,0.35,0.03,6.76,91.95


===== temperature =====


,year_month,year,month,region,temp_anomaly_c,temp_anomaly_baseline,co2_ppm
0,2000-01-01,2000,1,Global,0.432,1951-1980,367.44
1,2000-01-01,2000,1,Arctic,1.500,1951-1980,NaN
2,2000-01-01,2000,1,Northern_Hemisphere,0.468,1951-1980,NaN


In [7]:
prix_carbone.groupby("market").agg(
    devise=("currency", "unique"),
    debut=("date", "min"),
    fin=("date", "max"),
    jours=("price", "size"),
    prix_min=("price", "min"),
    prix_max=("price", "max"),
)

,devise,debut,fin,jours,prix_min,prix_max
market,,,,,,
California,[USD],2013-11-19,2026-04-30,3248,9.49,37.82
China_ETS,[CNY],2021-07-16,2026-04-30,1250,45.44,119.52
EU_ETS,[EUR],2005-04-22,2026-04-30,5485,0.05,104.04
RGGI,[USD],2008-09-25,2026-04-30,4591,1.79,22.22
UK_ETS,[GBP],2021-05-19,2026-04-30,1292,33.09,99.21


In [8]:
for nom in ["evenements", "emissions", "temperature"]:
    print(nom, "->", sorted(donnees[nom]["region"].unique()))

evenements -> ['Antarctica', 'Asia', 'Atlantic', 'Australia', 'Brazil', 'Canada', 'Caribbean', 'China', 'Europe', 'Greece', 'Hawaii', 'India', 'Japan', 'Libya', 'Nepal', 'Pakistan', 'Philippines', 'Russia', 'Siberia', 'Spain', 'Turkey', 'UAE', 'USA', 'global']
emissions -> ['Africa', 'Asia', 'Eurasia', 'Europe', 'LatAm', 'MENA', 'North America', 'Oceania']
temperature -> ['Arctic', 'Asia', 'Europe', 'Global', 'North_America', 'Northern_Hemisphere', 'Southern_Hemisphere', 'Tropics']


In [18]:
print(prix_carbone["market"].value_counts(), "\n")
print(evenements["event_type"].value_counts(), "\n")
print(emissions["region"].value_counts(), "\n")
print(temperature["region"].value_counts(), "\n")
print("Pays dans emissions :", emissions["country"].nunique())

market
EU_ETS        5485
RGGI          4591
California    3248
UK_ETS        1292
China_ETS     1250
Name: count, dtype: int64 

event_type
policy           11
hurricane         7
flood             7
wildfire          7
heat              4
earthquake        3
temperature       3
heatwave          2
tsunami           2
heatwave_fire     1
typhoon           1
winter_storm      1
drought           1
Name: count, dtype: int64 

region
Europe           378
Asia             297
MENA             270
LatAm            135
Africa           108
North America     81
Eurasia           54
Oceania           27
Name: count, dtype: int64 

region
Global                 316
Arctic                 316
Northern_Hemisphere    316
Southern_Hemisphere    316
Tropics                316
North_America          316
Europe                 316
Asia                   316
Name: count, dtype: int64 

Pays dans emissions : 50


## 3. Contrôles de cohérence
On cherche aussi les valeurs impossibles et incohérentes.

In [9]:
def bornes_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    ecart = q3 - q1
    return q1 - 1.5 * ecart, q3 + 1.5 * ecart


In [10]:
# Mix énergétique : parts cohérentes ?
parts = ["coal_pct", "oil_pct", "gas_pct", "nuclear_pct",
         "hydro_pct", "solar_pct", "wind_pct", "other_renewables_pct"]
somme_parts = mix_energie[parts].sum(axis=1)
print("Mix : somme des parts hors 99-101 :", ((somme_parts < 99) | (somme_parts > 101)).sum())
print("Mix : parts négatives ou > 100 :", ((mix_energie[parts] < 0) | (mix_energie[parts] > 100)).sum().sum())

Mix : somme des parts hors 99-101 : 0
Mix : parts négatives ou > 100 : 0


In [11]:
# Émissions : per capita cohérent avec émissions / population ?
ecart_per_capita = (emissions["co2_emissions_mt"] / emissions["population_millions"]
                    - emissions["co2_per_capita_t"]).abs()
print("Émissions : écart per capita > 0.1 :", (ecart_per_capita > 0.1).sum())
print("Émissions : doublons pays-année :", emissions.duplicated(subset=["country", "year"]).sum())


Émissions : écart per capita > 0.1 : 49
Émissions : doublons pays-année : 0


In [12]:
# Prix : valeurs impossibles et aberrantes par marché
print("Prix <= 0 :", (prix_carbone["price"] <= 0).sum())
for marche, groupe in prix_carbone.groupby("market"):
    bas, haut = bornes_iqr(groupe["price"])
    n_aberrants = ((groupe["price"] < bas) | (groupe["price"] > haut)).sum()
    print(f"Prix aberrants {marche} : {n_aberrants}")

Prix <= 0 : 0
Prix aberrants California : 0
Prix aberrants China_ETS : 0
Prix aberrants EU_ETS : 544
Prix aberrants RGGI : 296
Prix aberrants UK_ETS : 0


In [13]:
# Événements et température
print("Gravité min/max :", evenements["severity_score"].min(), evenements["severity_score"].max())
print("Température : doublons région-mois :", temperature.duplicated(subset=["region", "year_month"]).sum())
print("Anomalie min/max :", temperature["temp_anomaly_c"].min(), temperature["temp_anomaly_c"].max())

Gravité min/max : 6 10
Température : doublons région-mois : 0
Anomalie min/max : 0.219 4.522


In [14]:
# Les écarts per capita
anormales = emissions[ecart_per_capita > 0.1].assign(ecart=ecart_per_capita)
print(anormales["ecart"].describe())
print(anormales["country"].value_counts().head(10))
print(anormales[["country", "year", "co2_emissions_mt", "population_millions",
                 "co2_per_capita_t", "ecart"]].sort_values("ecart", ascending=False).head(10))


count    49.000000
mean      0.243358
std       0.152390
min       0.100000
25%       0.121429
50%       0.175000
75%       0.346667
max       0.624138
Name: ecart, dtype: float64
country
Qatar      26
Kuwait     16
UAE         3
Norway      2
Finland     2
Name: count, dtype: int64
    country  year  co2_emissions_mt  population_millions  co2_per_capita_t  \
879   Qatar  2015             108.1                  2.9             37.90   
874   Qatar  2010              95.8                  2.8             34.81   
889   Qatar  2025             114.3                  3.1             37.44   
884   Qatar  2020             102.1                  3.0             34.59   
869   Qatar  2005              73.9                  2.7             27.89   
888   Qatar  2024             122.2                  3.0             40.30   
878   Qatar  2014             104.5                  2.8             36.89   
864   Qatar  2000              54.8                  2.5             21.49   
883   Qatar  2

In [15]:
# Les prix aberrants : à quelles années ?
for marche in ["EU_ETS", "RGGI"]:
    prix_marche = prix_carbone[prix_carbone["market"] == marche]
    bas, haut = bornes_iqr(prix_marche["price"])
    hors_bornes = prix_marche[(prix_marche["price"] < bas) | (prix_marche["price"] > haut)]
    print(f"\n{marche} : bornes {bas:.1f} à {haut:.1f}")
    print(hors_bornes.groupby("year")["price"].agg(["size", "min", "max"]))


EU_ETS : bornes -36.9 à 80.2
      size    min     max
year                     
2021     5  80.78   82.45
2022   190  80.38  104.04
2023   250  80.59  103.60
2024    13  80.28   82.68
2025    14  80.16   84.24
2026    72  80.30   88.11

RGGI : bornes -6.3 à 20.2
      size    min    max
year                    
2024    56  20.26  21.79
2025   203  20.26  22.22
2026    37  20.25  21.21


## 4. Constats
1. Les 5 fichiers ne contiennent aucun doublon. Seul `co2_ppm` a des manquants (2 212 lignes), car il n'est renseigné que pour la région « Global ».
2. Les noms de régions diffèrent selon les fichiers (par exemple `North America` et `North_America`) : à harmoniser avant toute jointure.
3. Les prix du carbone sont dans des devises différentes : on comparera des variations en %, pas des niveaux.
4. 49 lignes d'émissions ont un `co2_per_capita_t` incohérent avec émissions / population : [À compléter : concentrées sur quels pays, quels écarts, décision].
5. EU ETS (544 lignes) et RGGI (296 lignes) ressortent comme aberrants selon la règle des 1,5 écart interquartile : [À compléter : concentrées sur quelles années, décision].

In [19]:
population_implicite = emissions["co2_emissions_mt"] / emissions["co2_per_capita_t"]
ecart_population = (population_implicite - emissions["population_millions"]).abs()
sur_les_49 = ecart_population[ecart_per_capita > 0.1]
print("Écart maximal sur la population, pour les 49 lignes (millions) :", round(sur_les_49.max(), 3))

Écart maximal sur la population, pour les 49 lignes (millions) : 0.05


## 4. Constats
1. Aucun doublon dans les 5 fichiers, y compris sur les clés (pays-année, région-mois). Seul `co2_ppm` a des manquants (2 212 lignes sur 2 528), car il n'est renseigné que pour la région « Global » : c'est un manquant structurel, on ne l'impute pas.
2. Les noms de régions diffèrent selon les fichiers : seuls « Asia » et « Europe » sont communs, et `North America` (émissions) s'écrit `North_America` (température). Les régions des événements mélangent continents, pays et océans. Les événements seront donc joints sur l'année, pas sur la région.
3. Chaque marché du carbone a une seule devise (USD, CNY, EUR, GBP), mais les niveaux de prix ne sont pas comparables entre marchés : on comparera des variations en % ou un indice base 100. Les marchés démarrent à des dates différentes (de 2005 à 2021) et tous s'arrêtent au 30/04/2026 : l'année 2026 est incomplète.
4. 49 lignes d'émissions ont un `co2_per_capita_t` qui s'écarte de émissions / population (de 0,10 à 0,62 t). Elles concernent des pays à petite population (Qatar 26, Kuwait 16, UAE 3, Norway 2, Finland 2). La population est arrondie à 0,1 million, ce qui explique ces écarts [à confirmer avec la cellule de vérification]. Décision : conserver `co2_per_capita_t`.
5. Selon la règle des 1,5 écart interquartile, l'EU ETS compte 544 prix aberrants (tous au-dessus de 80,2 €, dont 440 en 2022-2023) et RGGI 296 (entre 20,25 et 22,22 $, en 2024-2026). Ce sont des hausses réelles d'un marché non stationnaire, pas des erreurs : ils sont conservés.